# Autoencoders and the VAE

Every network so far predicted a label. This one learns to rebuild its own input through a narrow bottleneck, and in doing so learns a compact code for the data. A plain autoencoder gives a code that works for reconstruction but is a poor map for creating new images. A variational autoencoder (VAE) adds one idea that makes the map smooth enough to sample from. Both are trained here on 5,000 MNIST digits with a 2-number latent space, so the code can be plotted.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from common.datasets import load_mnist

SEED = 42
set_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
device = get_device()
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
X_tr = torch.from_numpy(x_tr.reshape(len(x_tr), -1))
X_te = torch.from_numpy(x_te.reshape(len(x_te), -1))
print("device:", device, "| train", tuple(X_tr.shape))

## 1. Intuition

An **autoencoder** has an encoder that squeezes 784 pixels to 2 numbers and a decoder that rebuilds the 784 pixels. The loss is how far the rebuild is from the original. To succeed, the two numbers must hold what matters about the digit.

Nothing makes the 2D code tidy. The encoder can scatter digits anywhere and leave holes the decoder never sees. A point in a hole decodes to rubbish. A **VAE** fixes this by encoding each image as a small cloud, not a point, and by pulling all the clouds towards one standard normal. Then any point near the origin decodes to something digit-like, and you can sample new digits by drawing from $\mathcal N(0,I)$.

## 2. Maths and cost function

The VAE encoder outputs a mean $\boldsymbol\mu$ and a log-variance $\log\boldsymbol\sigma^2$. It samples

$$\mathbf z=\boldsymbol\mu+\boldsymbol\sigma\odot\boldsymbol\epsilon,\qquad \boldsymbol\epsilon\sim\mathcal N(0,I)$$

This is the **reparameterisation trick**. Sampling is not differentiable, but this form moves the randomness into $\boldsymbol\epsilon$, which needs no gradient, so backprop flows through $\boldsymbol\mu$ and $\boldsymbol\sigma$. The loss has two parts:

$$L=\underbrace{-\log p(\mathbf x\mid\mathbf z)}_{\text{reconstruction}}+\beta\,\underbrace{\tfrac12\sum_j\big(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1\big)}_{\mathrm{KL}\big(q(\mathbf z\mid\mathbf x)\,\|\,\mathcal N(0,I)\big)}$$

Reconstruction is binary cross-entropy summed over the 784 pixels. The KL term is the price of straying from the standard normal. With $\beta=1$ this is the evidence lower bound. The two terms pull against each other.

## 3. From scratch

One class covers both models: with `variational=False` the encoder outputs a point and there is no KL term. The loop is the hand-written PyTorch kind from notebook 08, with the loss split in two so we can watch both parts.

In [ ]:
class Autoencoder(nn.Module):
    """Squeeze 784 pixels through ``latent`` numbers and rebuild them."""

    def __init__(self, latent: int = 2, variational: bool = False) -> None:
        super().__init__()
        self.variational = variational
        self.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU())
        self.mu = nn.Linear(256, latent)
        self.logvar = nn.Linear(256, latent) if variational else None
        self.dec = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784))

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        h = self.enc(x)
        return self.mu(h), (self.logvar(h) if self.variational else None)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
        mu, logvar = self.encode(x)
        z = mu
        if logvar is not None:  # the reparameterisation trick
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.dec(z), mu, logvar


def loss_terms(x: torch.Tensor, recon: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor | None):
    """Reconstruction (summed over pixels) and KL to N(0, I), both averaged over the batch."""
    rec = nn.functional.binary_cross_entropy_with_logits(recon, x, reduction="none").sum(1).mean()
    kl = (
        torch.zeros(())
        if logvar is None
        else (-0.5 * (1 + logvar - mu**2 - logvar.exp()).sum(1)).mean()
    )
    return rec, kl.to(rec.device)


def fit(
    model: Autoencoder, epochs: int = 12, beta: float = 1.0, seed: int = SEED
) -> list[tuple[float, float]]:
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    hist = []
    for _ in range(epochs):
        order = torch.randperm(len(X_tr), generator=g)
        tot_r = tot_k = 0.0
        for i in range(0, len(order), 100):
            xb = X_tr[order[i : i + 100]].to(device)
            rec, kl = loss_terms(xb, *model(xb))
            opt.zero_grad()
            (rec + beta * kl).backward()
            opt.step()
            tot_r, tot_k = tot_r + rec.item() * len(xb), tot_k + kl.item() * len(xb)
        hist.append((tot_r / len(order), tot_k / len(order)))
    return hist


def evaluate(model: Autoencoder) -> tuple[float, float]:
    model.eval()
    with torch.no_grad():
        return tuple(v.item() for v in loss_terms(X_te.to(device), *model(X_te.to(device))))


set_seed(SEED)
ae = Autoencoder(2)
h = fit(ae)
print(
    f"autoencoder: train reconstruction loss {h[0][0]:.0f} -> {h[-1][0]:.0f}, test {evaluate(ae)[0]:.0f}"
)

The autoencoder cuts the training reconstruction loss from 267 to 166 nats (summed over 784 pixels) in 12 epochs, and scores 164 on the test set.

## 4. Library version

The same loop with `variational=True`.

In [ ]:
set_seed(SEED)
vae = Autoencoder(2, variational=True)
h = fit(vae)
rec, kl = evaluate(vae)
print(f"VAE: test reconstruction {rec:.0f}, KL {kl:.1f}, total {rec + kl:.0f}")


def latent_plot(model: Autoencoder, ax: plt.Axes, title: str) -> None:
    model.eval()
    with torch.no_grad():
        z = model.encode(X_te.to(device))[0].cpu().numpy()
    ax.scatter(z[:, 0], z[:, 1], c=y_te, cmap="tab10", s=6)
    ax.set_title(title)


for name, m in (("autoencoder", ae), ("VAE", vae)):
    z = m.encode(X_te.to(device))[0].detach().cpu().numpy()
    print(
        f"{name}: latent std per dimension {z.std(0)}, 99th percentile of |z| {np.percentile(np.abs(z), 99):.1f}"
    )
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
latent_plot(ae, ax[0], "autoencoder latent space")
latent_plot(vae, ax[1], "VAE latent space")
plt.show()

The VAE reconstructs about as well (166) and pays 5.2 nats of KL. The total test loss is 172. The latent plots show the difference. The autoencoder's codes spread over a range of about $\pm10$ (99% of values below 10 in absolute value) with an irregular outline. The VAE's codes sit in a tighter blob (99% below 4) centred near the origin, with a standard deviation of 1.7 and 1.1 on the two axes. That is closer to the $\mathcal N(0,I)$ target, though not equal to it. In both plots the zeros and ones separate and the other digits overlap in the middle. The labels were never used in training.

Now decode a 10x10 grid of latent points. For the autoencoder the grid spans the range its codes use. For the VAE it spans $[-2.5, 2.5]$.

In [ ]:
def decode_grid(model: Autoencoder, lim: float, n: int = 10) -> np.ndarray:
    zs = np.linspace(-lim, lim, n)
    z = torch.tensor([[a, b] for b in zs[::-1] for a in zs], dtype=torch.float32, device=device)
    with torch.no_grad():
        imgs = torch.sigmoid(model.dec(z)).cpu().numpy().reshape(n, n, 28, 28)
    return imgs.transpose(0, 2, 1, 3).reshape(n * 28, n * 28)


zs_ae = ae.encode(X_te.to(device))[0].detach().cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(decode_grid(ae, np.abs(zs_ae).max() * 0.8), cmap="gray")
ax[0].set_title("autoencoder: decode a grid of latent points")
ax[1].imshow(decode_grid(vae, 2.5), cmap="gray")
ax[1].set_title("VAE: decode a grid over [-2.5, 2.5]")
for a in ax:
    a.axis("off")
plt.show()
set_seed(SEED)
z = torch.randn(5, 2, device=device)
print(
    "five samples drawn from N(0, I) decoded by the VAE: pixel mean",
    torch.sigmoid(vae.dec(z)).mean().item(),
)

The autoencoder's grid has broken, fragmentary shapes in the top left, where few training codes landed, and sharp digits where they did. The VAE grid morphs smoothly from a 0 through 7s and 1s to 6s and 9s with no junk cells, though everything is blurrier. Smooth and blurry is the VAE bargain.

## 5. Tuning

$\beta$ weights the KL term. Five values, 8 epochs each.

In [ ]:
for beta in (0.0, 0.1, 1.0, 10.0, 100.0):
    set_seed(SEED)
    m = Autoencoder(2, variational=True)
    fit(m, epochs=8, beta=beta)
    rec, kl = evaluate(m)
    print(f"beta {beta:>5}: test reconstruction {rec:6.1f}, KL {kl:6.2f}")

At $\beta=0$ there is no pull and the KL is 36.7 nats. As $\beta$ rises the KL falls (16.6, 5.7, 1.5, 0.0) and the reconstruction loss rises from 168 to 206. That is the trade in numbers: a code closer to the prior is easier to sample and worse at reconstruction.

## 6. What breaks

At $\beta=100$ the KL is 0.00. The encoder has learned to ignore the image: every input maps to the same cloud, the standard normal. This is **posterior collapse**. Compare the codes at $\beta=1$ and $\beta=100$.

In [ ]:
for beta in (1.0, 100.0):
    set_seed(SEED)
    m = Autoencoder(2, variational=True)
    fit(m, epochs=8, beta=beta)
    with torch.no_grad():
        mu = m.encode(X_te.to(device))[0]
        out = torch.sigmoid(m.dec(mu)).cpu().numpy()
    print(
        f"beta {beta}: latent std per dimension {mu.std(0).cpu().numpy()}, spread of reconstructions {out.std(0).mean():.3f}"
    )

At $\beta=100$ the latent means have a standard deviation of 0.004 and 0.005, against 1.8 and 1.1 at $\beta=1$. Every test digit reconstructs to the same image (the spread of the outputs across images is 0.000, against 0.081). The decoder is left with only the average digit. Collapse can also happen at $\beta=1$ with a powerful decoder. Is the KL near zero while the reconstructions look good? Then the latent is being ignored.